In [2]:
# from kaggle_secrets import UserSecretsClient
# from huggingface_hub import login
# from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# try:
#     user_secrets = UserSecretsClient()
#     hf_token = user_secrets.get_secret("HF_TOKEN")
#     print("Logging into Hugging Face...")
#     login(token=hf_token)
# except Exception as e:
#     print("Error: Could not find the HF_TOKEN in Kaggle Secrets.")
#     raise e

# HF_USERNAME = "alokkohli200" 


# REPO_NAME = f"{HF_USERNAME}/Alok_ATML_project"

# # 3. Paste the exact path you copied in Step 3
# LOCAL_MODEL_DIR = "/kaggle/input/notebooks/alokkohli200/advanced-ml-epoch-2/distilbart_arxiv_hybrid" 

# print(f"Loading model and tokenizer from {LOCAL_MODEL_DIR}...")
# model = AutoModelForSeq2SeqLM.from_pretrained(LOCAL_MODEL_DIR)
# tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_DIR)

# print(f"Pushing model to https://huggingface.co/{REPO_NAME} ...")
# model.push_to_hub(REPO_NAME)

# print("Pushing tokenizer...")
# tokenizer.push_to_hub(REPO_NAME)

# print("Success! Your model is now securely hosted.")

Logging into Hugging Face...
Loading model and tokenizer from /kaggle/input/notebooks/alokkohli200/advanced-ml-epoch-2/distilbart_arxiv_hybrid...


Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/359 [00:00<?, ?it/s]

Pushing model to https://huggingface.co/alokkohli200/Alok_ATML_project ...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushing tokenizer...


README.md: 0.00B [00:00, ?B/s]

Success! Your model is now securely hosted.


In [3]:
!pip install python-docx transformers scikit-learn nltk

# ==========================================
# Step 2: Import Libraries
# ==========================================
import torch
import docx
import re
import nltk
import numpy as np
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.tokenize import sent_tokenize

# Download NLTK data
nltk.download('punkt')
nltk.download('punkt_tab')

# ==========================================
# Step 3: Text Extraction & Preprocessing Logic
# ==========================================
def read_docx(file_path):
    """Extracts all text from a Word document, ignoring images and formatting."""
    doc = docx.Document(file_path)
    full_text = []
    for para in doc.paragraphs:
        if para.text.strip() != "":
            full_text.append(para.text.strip())
    return "\n".join(full_text)

def extract_sections(text):
    """
    Intelligently splits the text into Intro, Methodology, and Conclusion
    by looking for common academic headings.
    """
    # Regex to find sections based on common headings
    intro_match = re.search(r'(Introduction.*?)(Methodology|Methods|Experimental Setup|Results)', text, re.IGNORECASE | re.DOTALL)
    conc_match = re.search(r'(Conclusion.*?)(Future Scope|References|Appendix|$)', text, re.IGNORECASE | re.DOTALL)

    if intro_match and conc_match:
        intro = intro_match.group(1)
        conclusion = conc_match.group(1)
        
        # Methodology is everything between Intro and Conclusion
        body_start = intro_match.end(1)
        body_end = conc_match.start(1)
        methodology = text[body_start:body_end]
        
        return {"intro": intro, "methodology": methodology, "conclusion": conclusion}
    else:
        # Fallback to the 20/60/20 heuristic if headings aren't found
        sentences = sent_tokenize(text)
        total_len = len(sentences)
        return {
            "intro": " ".join(sentences[:int(total_len * 0.2)]),
            "methodology": " ".join(sentences[int(total_len * 0.2):int(total_len * 0.8)]),
            "conclusion": " ".join(sentences[int(total_len * 0.8):])
        }

def get_top_tfidf_sentences(text, max_tokens, tokenizer):
    """Extracts top sentences using TF-IDF until the token budget is reached."""
    sentences = sent_tokenize(text)
    if not sentences:
        return ""
    
    vectorizer = TfidfVectorizer(stop_words='english')
    try:
        tfidf_matrix = vectorizer.fit_transform(sentences)
    except ValueError:
        return text 
        
    sentence_scores = np.array(tfidf_matrix.sum(axis=1)).flatten()
    ranked_indices = sentence_scores.argsort()[::-1]
    
    selected_indices = []
    current_tokens = 0
    
    for idx in ranked_indices:
        sentence = sentences[idx]
        tokens = len(tokenizer.tokenize(sentence))
        if current_tokens + tokens <= max_tokens:
            selected_indices.append(idx)
            current_tokens += tokens
        else:
            break
            
    selected_indices.sort()
    return " ".join([sentences[i] for i in selected_indices])

def prepare_paper_for_model(raw_text, tokenizer):
    """Applies the dynamic 400-400-200 TF-IDF extraction to a single paper."""
    sections = extract_sections(raw_text)
    
    intro_tokens = len(tokenizer.tokenize(sections["intro"]))
    meth_tokens = len(tokenizer.tokenize(sections["methodology"]))
    conc_tokens = len(tokenizer.tokenize(sections["conclusion"]))
    
    # Base Budgets
    intro_budget, meth_budget, conc_budget = 400, 400, 200
    
    # Dynamic Rollover: Pass unused tokens to the methodology section
    if intro_tokens < intro_budget:
        meth_budget += (intro_budget - intro_tokens)
        intro_budget = intro_tokens
        
    if conc_tokens < conc_budget:
        meth_budget += (conc_budget - conc_tokens)
        conc_budget = conc_tokens
        
    best_intro = get_top_tfidf_sentences(sections["intro"], intro_budget, tokenizer)
    best_meth = get_top_tfidf_sentences(sections["methodology"], meth_budget, tokenizer)
    best_conc = get_top_tfidf_sentences(sections["conclusion"], conc_budget, tokenizer)
    
    return f"Introduction: {best_intro} Methodology: {best_meth} Conclusion: {best_conc}"

# ==========================================
# Step 4: Model Initialization & Inference
# ==========================================

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load your custom trained model directly from Hugging Face!
MODEL_NAME = "alokkohli200/Alok_ATML_project"
print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

# ---------------------------------------------------------
# IMPORTANT: Upload your document to Kaggle/Colab and 
# replace 'your_document.docx' with the actual file name.
# ---------------------------------------------------------



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 9.4 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Using device: cpu
Loading alokkohli200/Alok_ATML_project...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/359 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.84G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/359 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/985 [00:00<?, ?B/s]

In [5]:
FILE_PATH = "/kaggle/input/datasets/alokkohli200/alok-progress-report-2-scientific-paper/Alok_progress_report_2_Scientific_Paper.docx" 

print("Reading and preprocessing document...")
raw_paper_text = read_docx(FILE_PATH)
distilled_input = prepare_paper_for_model(raw_paper_text, tokenizer)

print("Generating abstractive summary...")
inputs = tokenizer(
    distilled_input, 
    max_length=1024, 
    truncation=True, 
    return_tensors="pt"
).to(device)

# Generate the summary
summary_ids = model.generate(
    inputs["input_ids"], 
    max_length=256,       # Max length of the generated abstract
    min_length=50,        # Ensure it doesn't just output one sentence
    length_penalty=2.0,   # Encourages slightly longer, more complete sentences
    num_beams=4,          # Beam search for better grammar and flow
    early_stopping=True
)

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("")
print("FINAL ABSTRACTIVE SUMMARY:")
print("")
print(summary)



Reading and preprocessing document...
Generating abstractive summary...

FINAL ABSTRACTIVE SUMMARY:

this paper presents a unique challenge in natural language processing , due to the extreme length of the papers , dense research papers , and the presence of complex applications . 
 the goal of this paper is to develop an accurate algorithm for constructing a simple and accurate summarizer , a sequence- to - sequence transform model , which is a sequence - to - sequences transform model . in this paper 
 , we present the first results from a simulation of this model on the ccdv/arxiv- summedization database , which consists of 7,000 research papers coupled with their author-spaces .    * key words and phrases * : numerical , numerical , and numerical methods for computing the model .
